<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Building Vectors from PDFs
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style="font-size:24px;font-family:Arial;color:#00233C"><b>Introduction</b></p>

<p style="font-size:16px;font-family:Arial;color:#00233C">
This notebook demonstrates how to create and store embeddings from a pdf in a table in Teradata Vantage. Once the pdf is parsed, the content can be stored, embeddings created and used entirely in SQL.
</p>

<p style="font-size:24px;font-family:Arial;color:#00233C"><b>Notebook Workflow Steps</b></p>

<div style="font-size:16px;font-family:Arial;color:#00233C">
<ol>
<li>
<b>Partition the pdf</b>
<ul>
<li>Import the required libraries</li>
<li>Use unstructured.io to partition the given pdf</li>
<li>Convert the list of pdf elements into a dictionary of elements</li>
</ul>
</li>

<li>
<b>Store the pdf elements and generate embeddings using Teradata Vector tables</b>
<ul>
<li>Connect to Vantage</li>
<li>Load the pdf elements into a table in Vantage</li>
<li>Generate embeddings using AWS embedding model and store in Vector datatype in table</li>
</ul>
</li>

## Using Unstructured.io to parse the pdf file into a table

### Import the required libraries

In [2]:
# Unstructured libraries
from unstructured.partition.image import partition_image
from unstructured.partition.pdf import partition_pdf

# For the Embeddigns
import sys
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
import pandas as pd
import json

# Teradata Vector Store libraries
from getpass import getpass
from teradatagenai import VSManager, VectorStore, VSPattern, VSApi
from teradataml import create_context, set_auth_token, execute_sql, display, DataFrame
display.max_rows = 100000

import numpy as np
import re


### Partition the pdf using Unstructured.io

In [3]:
#Partition the PDF file into chunks
pdfpath = r"DensePassageRetrieval.pdf"

# Partition the PDF document
elements = partition_pdf(
    filename=pdfpath,                  # mandatory
    strategy="hi_res",                                     # mandatory to use ``hi_res`` strategy
    extract_images_in_pdf=True,                            # mandatory to set as ``True``          
    extract_image_block_to_payload=False,                  # optional
    )

### Store the partitioned elements in a dictionary

In [4]:
#Convert the list of elements to a list of dictionaries
element_dicts = [element.to_dict() for element in elements]

# save the list locally:
file = r"UnstructuredDemoJSON.json"
with open(file, "w") as file:
    json.dump(element_dicts, file, indent=2)

## Store the pdf elements and generate embeddings using Teradata Vector tables

### Connect to Vantage

In [5]:
# Connect to Vantage using create_context.
hostname = getpass(prompt = 'hostname: ')
username = getpass(prompt = 'username: ')
password = getpass(prompt = 'password: ')

context=create_context(host=hostname, username=username, password=password)

### Load the partitioned text into a table in Vantage

In [6]:
#Create a table to store the PDF elements
execute_sql("""
CREATE MULTISET TABLE TD_Vector.pdf_elements (
    id INTEGER GENERATED ALWAYS AS IDENTITY NOT NULL,
    text VARCHAR(10000),
    PRIMARY KEY (id)
);
""")

TeradataCursor uRowsHandle=9 bClosed=False

In [7]:
#Clean data function
import re
def clean_data(data):
    # Remove non-ASCII characters
    cleanData = re.sub(r'[^\x00-\x7F]+', '', data)
    return cleanData.replace('·', '*').replace("'", "")

In [9]:
# Insert the pdf elements into the Teradata table
for element in elements:
    if element.text:
        execute_sql(f"""
        INSERT INTO TD_Vector.pdf_elements (text) VALUES ('{clean_data(element.text)}');
        """)

### Create embeddings in SQL for the chunked pdf and store in Vector datatype

In [28]:
#Create embeddings from the pdf elements
execute_sql("""
CREATE TABLE pdf_embeddings AS (
    SELECT * FROM AI_TextEmbeddings(
        ON pdf_elements AS InputTable
        USING
        authorization(AWSEmbeddingsAuth)
        region('us-west-2')
        apitype('aws')
        modelname('amazon.titan-embed-text-v1')
        textcolumn('text')
        outputformat('vector')
    ) as DT
) WITH DATA;   
""")

TeradataCursor uRowsHandle=456 bClosed=False

In [45]:
#View text and embeddings
df_embeddings = DataFrame.from_query("SELECT * FROM pdf_embeddings WHERE id >= 20;")
df_embeddings.head(10)

id,text,Embedding,Message
22,1The code and trained models have been released at https://github.com/facebookresearch/DPR.,"1.25,0.808594,0.535156,-0.486328,0.273438,0.0258789,-0.126953,1.77622e-05,0.168945,-0.433594,0.570312,0.090332,-0.065918,0.181641,-0.332031,-0.172852,0.119629,0.652344,-0.00205994,0.294922,-0.429688,-0.0050354,-0.289062,0.177734,0.546875,0.523438,-0.287109,0.043457,-0.0334473,0.199219,-0.558594,-0.777344,0.792969,0.105957,0.0375977,-0.0800781,0.121582,0.800781,0.46875,0.147461,-0.0283203,0.0437012,-0.359375,-0.570312,-0.026123,-0.412109,-0.152344,0.339844,0.419922,-0.118164,0.5,0.605469,0.0132446,0.00564575,-0.211914,0.287109,-0.0620117,-0.304688,-0.308594,-0.0654297,0.0515137,-0.157227,-0.376953,0.457031,-0.251953,0.457031,0.0556641,-0.652344,-0.192383,0.142578,0.103516,-0.386719,0.0708008,0.375,0.703125,0.125977,-0.0913086,0.289062,-0.0368652,-0.601562,-0.0388184,0.375,0.0214844,0.318359,0.683594,0.0250244,-0.132812,0.363281,0.000134468,-0.0050354,-0.460938,-0.0476074,-0.101562,0.259766,-0.0071106,-0.722656,-0.322266,-0.365234,-0.00338745,-0.419922,-0.0371094,-0.130859,-0.182617,-0.225586,-0.186523,0.093261",
24,Retrieval in open-domain QA is usually imple-,"0.339844,0.453125,0.28125,-0.460938,-1.00781,-0.496094,-0.00946045,-0.000915527,-0.400391,0.10791,0.78125,-0.129883,0.0615234,0.386719,-0.863281,0.217773,0.5625,0.25,-0.660156,0.164062,-0.660156,-0.00744629,-0.527344,0.800781,0.503906,0.664062,0.0270996,0.304688,0.433594,-0.143555,-0.19043,0.519531,0.960938,-0.196289,0.3125,0.220703,0.449219,0.769531,-0.271484,0.296875,0.396484,0.198242,-0.5,-0.0554199,-0.0463867,0.000406265,0.330078,0.135742,-0.484375,0.237305,-0.0405273,0.423828,-0.519531,0.232422,-0.036377,-0.105469,-0.0766602,-0.135742,-0.402344,0.0849609,-0.106934,0.523438,-0.53125,-0.804688,0.0864258,0.332031,0.0157471,-0.410156,0.558594,0.287109,-0.132812,-0.59375,0.373047,-0.738281,1.17188,0.527344,-0.193359,0.419922,-0.12793,-0.5625,-0.667969,0.486328,0.178711,-0.0480957,-0.542969,-0.213867,-0.777344,0.074707,0.000123024,0.373047,-0.226562,0.570312,0.163086,0.0554199,0.753906,-0.470703,-1.07031,0.0175781,0.679688,-0.515625,-0.0255127,0.131836,-0.0341797,0.0229492,-0.462891,0.133789,-0.328125,-0.20703",
25,"mented using TF-IDF or BM25 (Robertson and Zaragoza, 2009), which matches keywords ef- ciently with an inverted index and can be seen as representing the question and context in high- dimensional, sparse vectors (with weighting). Con- versely, the dense, latent semantic encoding is com- plementary to sparse representations by design. For example, synonyms or paraphrases that consist of completely different tokens may still be mapped to vectors close to each other. Consider the question Who is the bad guy in lord of the rings?, which can be answered from the context Sala Baker is best known for portraying the villain Sauron in the Lord of the Rings trilogy. A term-based system would have difculty retrieving such a context, while a dense retrieval system would be able to better match bad guy with villain and fetch the cor- rect context. Dense encodings are also learnable by adjusting the embedding functions, which pro- vides additional exibility to have a task-specic representation. With special in-memory data struc- tures and indexing schemes, retrieval can be done efciently using maximum inner product search (MIPS) algorithms (e.g., Shrivastava and Li (2014); Guo et al. (2016)).","0.0541992,0.322266,0.490234,-0.0556641,-0.535156,0.0639648,-0.263672,0.000226974,0.162109,0.0498047,0.664062,0.132812,0.0142212,0.139648,-0.439453,0.183594,0.283203,0.110352,-0.326172,0.189453,-0.0290527,0.09375,-0.25,0.0603027,-0.115723,-0.0966797,0.160156,-0.0800781,0.141602,0.0703125,-0.128906,0.128906,0.431641,0.625,0.163086,-0.0471191,0.269531,0.155273,-0.341797,0.0795898,0.206055,-0.447266,-0.106934,-0.0742188,0.339844,0.139648,0.119141,0.24707,-0.177734,0.0334473,0.439453,-0.382812,0.0727539,0.349609,0.376953,-0.041748,-0.145508,0.298828,-0.570312,0.152344,

### Clean up

In [21]:
#Drop the tables
execute_sql("""
DROP TABLE pdf_elements;
""")
execute_sql("""
DROP TABLE pdf_embeddings;
""")

TeradataCursor uRowsHandle=250 bClosed=False